In [1]:
# Load .env

from dotenv import load_dotenv
import os

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")
DOWNLOAD_PATH = os.getenv("DOWNLOAD_PATH")

In [2]:
# Download yeanzc/telco-customer-churn-ibm-dataset from kaggle

import kaggle

kaggle.api.dataset_download_files(
    "yeanzc/telco-customer-churn-ibm-dataset", path=DOWNLOAD_PATH, unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset


In [3]:
# Load xlsx file into a pandas dataframe
import pandas as pd

df = pd.read_excel(os.path.join(DOWNLOAD_PATH, "Telco_customer_churn.xlsx"))
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
# Show column names and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

In [5]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

constraint_queries = [
    "CREATE CONSTRAINT customer_id IF NOT EXISTS FOR (c:Customer) REQUIRE c.CustomerID IS UNIQUE;",
    "CREATE CONSTRAINT location_zip IF NOT EXISTS FOR (l:Location) REQUIRE l.`Zip Code` IS UNIQUE;",
    "CREATE CONSTRAINT payment_method IF NOT EXISTS FOR (p:PaymentMethod) REQUIRE p.`Payment Method` IS UNIQUE;",
    "CREATE CONSTRAINT contract_type IF NOT EXISTS FOR (c:Contract) REQUIRE c.Contract IS UNIQUE;",
    "CREATE CONSTRAINT service_type IF NOT EXISTS FOR (s:Service) REQUIRE s.ServiceType IS UNIQUE;",
]


def apply_constraints(tx):
    for query in constraint_queries:
        tx.run(query)


with driver.session(database=NEO4J_DATABASE) as session:
    session.execute_write(apply_constraints)

driver.close()
print("Neo4j database constraints and indexes created successfully!")

Neo4j database constraints and indexes created successfully!


In [6]:
# Initialize PySpark with the Neo4j Connector
from pyspark.sql import SparkSession
import numpy as np

# Clean Pandas column names: lowercase, replace spaces with underscores, and rename 'customerid'
df_clean = df.copy()
df_clean.columns = [c.strip().replace(" ", "_").lower() for c in df_clean.columns]
df_clean.rename(columns={"customerid": "customer_id"}, inplace=True)

# 'total_charges' has blank spaces for customers with 0 tenure. Convert to 0.0 and cast to float.
df_clean["total_charges"] = pd.to_numeric(
    df_clean["total_charges"].replace(r"^\s*$", np.nan, regex=True)
).fillna(0.0)

# PySpark schema inference fails on string columns with float NaNs. Fill them explicitly.
df_clean["churn_reason"] = df_clean["churn_reason"].fillna("No Reason Provided")

# Initialize Spark
spark = (
    SparkSession.builder.appName("SkyTelcoGraphLoad")
    .config(
        "spark.jars.packages",
        "org.neo4j:neo4j-connector-apache-spark_2.12:5.3.0_for_spark_3",
    )
    .config("neo4j.url", NEO4J_URI)
    .config("neo4j.authentication.type", "basic")
    .config("neo4j.authentication.basic.username", NEO4J_USER)
    .config("neo4j.authentication.basic.password", NEO4J_PASSWORD)
    .config("neo4j.database", NEO4J_DATABASE)
    .getOrCreate()
)

# Create Spark DataFrame directly from the natively typed Pandas DataFrame
spark_df = spark.createDataFrame(df_clean)

print("Spark DataFrame created with native numeric types!")
# Print the schema so we can verify the types before loading into Neo4j
spark_df.printSchema()

/var/folders/y4/kvz3k9ln1zbb35kl20jvlpmm0000gp/T/ipykernel_8453/60925088.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean["total_charges"].replace(r"^\s*$", np.nan, regex=True)
Ivy Default Cache set to: /Users/pedroleitao/.ivy2/cache
The jars for the packages stored in: /Users/pedroleitao/.ivy2/jars
org.neo4j#neo4j-connector-apache-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9d16b96f-f5df-49c0-9ed4-3edaaf74520b;1.0
	confs: [default]
	found org.neo4j#neo4j-connector-apache-spark_2.12;5.3.0_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.12_common;5.3.0 in central


:: loading settings :: url = jar:file:/Users/pedroleitao/miniconda3/envs/customer360-churn/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.neo4j.driver#neo4j-java-driver;4.4.13 in central
	found org.reactivestreams#reactive-streams;1.0.4 in central
	found org.apache.xbean#xbean-asm6-shaded;4.10 in central
	found org.neo4j#neo4j-cypher-dsl;2022.9.1 in central
	found org.apiguardian#apiguardian-api;1.1.2 in central
:: resolution report :: resolve 114ms :: artifacts dl 5ms
	:: modules in use:
	org.apache.xbean#xbean-asm6-shaded;4.10 from central in [default]
	org.apiguardian#apiguardian-api;1.1.2 from central in [default]
	org.neo4j#neo4j-connector-apache-spark_2.12;5.3.0_for_spark_3 from central in [default]
	org.neo4j#neo4j-connector-apache-spark_2.12_common;5.3.0 from central in [default]
	org.neo4j#neo4j-cypher-dsl;2022.9.1 from central in [default]
	org.neo4j.driver#neo4j-java-driver;4.4.13 from central in [default]
	org.reactivestreams#reactive-streams;1.0.4 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            || 

Spark DataFrame created with native numeric types!
root
 |-- customer_id: string (nullable = true)
 |-- count: long (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: long (nullable = true)
 |-- lat_long: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- senior_citizen: string (nullable = true)
 |-- partner: string (nullable = true)
 |-- dependents: string (nullable = true)
 |-- tenure_months: long (nullable = true)
 |-- phone_service: string (nullable = true)
 |-- multiple_lines: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- online_security: string (nullable = true)
 |-- online_backup: string (nullable = true)
 |-- device_protection: string (nullable = true)
 |-- tech_support: string (nullable = true)
 |-- streaming_tv: string (nullable = true)
 |-- streaming_movi

In [7]:
print("Writing Core Nodes & Relationships to Neo4j...")

# Customer Nodes (Select ONLY customer properties)
customer_df = spark_df.select(
    "customer_id",
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "churn_label",
    "churn_score",
    "churn_reason",
    "cltv",
).distinct()

customer_df.write.format("org.neo4j.spark.DataSource").mode("Overwrite").option(
    "labels", "Customer"
).option("node.keys", "customer_id").save()

# Location Nodes & LIVES_AT Relationships
location_df = spark_df.select(
    "zip_code", "city", "state", "country", "latitude", "longitude", "lat_long"
).distinct()

location_df.write.format("org.neo4j.spark.DataSource").mode("Overwrite").option(
    "labels", "Location"
).option("node.keys", "zip_code").save()

# Write LIVES_AT Relationship (Selecting only the keys needed to draw the edge)
spark_df.select("customer_id", "zip_code").repartition(1).write.format(
    "org.neo4j.spark.DataSource"
).mode("Overwrite").option("relationship", "LIVES_AT").option(
    "relationship.save.strategy", "keys"
).option(
    "relationship.source.labels", "Customer"
).option(
    "relationship.source.node.keys", "customer_id"
).option(
    "relationship.target.labels", "Location"
).option(
    "relationship.target.node.keys", "zip_code"
).save()

# PaymentMethod Nodes & PAYS_WITH Relationships
payment_df = spark_df.select("payment_method").distinct()

payment_df.write.format("org.neo4j.spark.DataSource").mode("Overwrite").option(
    "labels", "PaymentMethod"
).option("node.keys", "payment_method").save()

spark_df.select("customer_id", "payment_method").repartition(1).write.format(
    "org.neo4j.spark.DataSource"
).mode("Overwrite").option("relationship", "PAYS_WITH").option(
    "relationship.save.strategy", "keys"
).option(
    "relationship.source.labels", "Customer"
).option(
    "relationship.source.node.keys", "customer_id"
).option(
    "relationship.target.labels", "PaymentMethod"
).option(
    "relationship.target.node.keys", "payment_method"
).save()

# Contract Nodes & HAS_CONTRACT Relationships
contract_df = spark_df.select("contract", "paperless_billing").distinct()

contract_df.write.format("org.neo4j.spark.DataSource").mode("Overwrite").option(
    "labels", "Contract"
).option("node.keys", "contract").save()

spark_df.select("customer_id", "contract").repartition(1).write.format(
    "org.neo4j.spark.DataSource"
).mode("Overwrite").option("relationship", "HAS_CONTRACT").option(
    "relationship.save.strategy", "keys"
).option(
    "relationship.source.labels", "Customer"
).option(
    "relationship.source.node.keys", "customer_id"
).option(
    "relationship.target.labels", "Contract"
).option(
    "relationship.target.node.keys", "contract"
).save()

print("Clean core schema successfully loaded!")

Writing Core Nodes & Relationships to Neo4j...


26/03/07 17:02:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Clean core schema successfully loaded!


In [8]:
from pyspark.sql.functions import expr, col

print("Melting and Writing Services...")

service_cols = [
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
]

stack_args = ", ".join([f"'{c}', `{c}`" for c in service_cols])
stack_expr = f"stack({len(service_cols)}, {stack_args}) as (service_type, has_service)"

services_df = spark_df.select("customer_id", expr(stack_expr))

active_services_df = services_df.filter(
    col("has_service").isin("Yes", "DSL", "Fiber optic")
)

# Write the unique Service Nodes
active_services_df.select("service_type").distinct().write.format(
    "org.neo4j.spark.DataSource"
).mode("Overwrite").option("labels", "Service").option(
    "node.keys", "service_type"
).save()

# Write the SUBSCRIBES_TO Relationships
active_services_df.select("customer_id", "service_type", "has_service").repartition(
    1
).write.format("org.neo4j.spark.DataSource").mode("Overwrite").option(
    "relationship", "SUBSCRIBES_TO"
).option(
    "relationship.save.strategy", "keys"
).option(
    "relationship.source.labels", "Customer"
).option(
    "relationship.source.node.keys", "customer_id"
).option(
    "relationship.target.labels", "Service"
).option(
    "relationship.target.node.keys", "service_type"
).option(
    "relationship.properties", "has_service"
).save()

print("Graph generation completely finished! Nodes are clean and formatted correctly.")

Melting and Writing Services...


Graph generation completely finished! Nodes are clean and formatted correctly.


In [9]:
from neo4j_analysis import Neo4jAnalysis
from neo4j_viz.neo4j import from_neo4j, ColorSpace

analysis = Neo4jAnalysis(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)

query = """
CALL db.schema.visualization()
"""

results = analysis.run_query_viz(query)

colors = {
    "Customer": "#FF5A5F",
    "Location": "#00A699",
    "PaymentMethod": "#FC642D",
    "Contract": "#484848",
    "Service": "#767676",
}

VG = from_neo4j(results)
VG.color_nodes(
    field="caption",  # Using the internal labels property
    color_space=ColorSpace.DISCRETE,
    colors=colors,
)

generated_html = VG.render(layout="forcedirected")
await analysis.capture_graph_to_png(
    generated_html, "renderings/schema_graph.png", width=1080
)

![Graph Schema](renderings/schema_graph.png)